# Phase 3 — HGT/HAN baselines on the same PrimeKG splits

Attention-based heterogeneous GNN baselines (PyG's `HGTConv` and `HANConv`), trained **single-stage** (no pretrain phase) directly on the indication/contraindication task — this is the actual architectural contrast with Phase 2's two-phase TxGNN-style model.

Loads `splits.npz` saved by Phase 2 so both baselines are evaluated on **exactly** the same drug-disease pairs as the TxGNN reimplementation. Same dot-product decoder as Phase 2, so the comparison isolates the encoder architecture.

**Scale reduced to fit a single T4 GPU**, after the first attempt OOM'd doing full-batch attention over PrimeKG's 8.1M edges: hidden dim 64→32, attention heads 4→2, each relation type capped at 50,000 edges (sampled, seed 42), mixed precision (fp16 autocast). This is documented in every result JSON's `notes` field — these numbers are not directly comparable to a full-scale run, only to each other and to Phase 2 if Phase 2 is rerun under the same constraints.

In [19]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/txgnn_comparison'
kg_path = f'{PROJECT_DIR}/data/kg.csv'
splits_path = f'{PROJECT_DIR}/data/splits.npz'
assert os.path.exists(kg_path), 'Run Phase 1 first.'
assert os.path.exists(splits_path), 'Run Phase 2 first (splits.npz not found) — Phase 3 must reuse Phase 2 splits.'
print('Found kg.csv and splits.npz')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found kg.csv and splits.npz


In [20]:
import pandas as pd, numpy as np, torch, json, time, random
import torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rng = np.random.RandomState(SEED)
HIDDEN = 32           # reduced from 64 — single T4 OOM'd on full-batch attention over 8.1M edges
MAX_EDGES_PER_RELATION = 50000  # cap each relation type; see capping cell below
print('Device:', device)

kg = pd.read_csv(kg_path, low_memory=False)
splits = np.load(splits_path)
n_drug, n_disease = int(splits['n_drug']), int(splits['n_disease'])
ind_train_r, ind_valid_r, ind_test_r = splits['ind_train_r'], splits['ind_valid_r'], splits['ind_test_r']
contra_train_r, contra_valid_r, contra_test_r = splits['contra_train_r'], splits['contra_valid_r'], splits['contra_test_r']
ind_train_z, ind_valid_z, ind_test_z = splits['ind_train_z'], splits['ind_valid_z'], splits['ind_test_z']
contra_train_z, contra_valid_z, contra_test_z = splits['contra_train_z'], splits['contra_valid_z'], splits['contra_test_z']
print('Loaded splits — random indication train/valid/test:', len(ind_train_r), len(ind_valid_r), len(ind_test_r))

Device: cuda
Loaded splits — random indication train/valid/test: 6571 1408 1409


In [21]:
# Rebuild the same heterogeneous graph as Phase 2 (deterministic from kg.csv,
# so this reproduces identical node-index mappings without needing to re-save them).
# Edges are capped per relation type (MAX_EDGES_PER_RELATION) — full-batch HGT/HAN
# attention over all 8.1M edges OOM'd a T4. This trades graph coverage for actually
# finishing; documented explicitly in every result JSON's notes field.
nodes_x = kg[['x_index', 'x_id', 'x_type', 'x_name']].rename(columns=lambda c: c[2:])
nodes_y = kg[['y_index', 'y_id', 'y_type', 'y_name']].rename(columns=lambda c: c[2:])
all_nodes = pd.concat([nodes_x, nodes_y]).drop_duplicates(subset=['index']).sort_values('index').reset_index(drop=True)

node_types = sorted(all_nodes['type'].unique())
local_idx = {}
type_counts = {}
for t in node_types:
    sub = all_nodes[all_nodes['type'] == t].sort_values('index')
    type_counts[t] = len(sub)
    for local_i, global_i in enumerate(sub['index'].values):
        local_idx[global_i] = (t, local_i)

edge_type_groups = kg.groupby(['x_type', 'relation', 'y_type'])
edge_index_by_type = {}
total_before, total_after = 0, 0
for (src_t, rel, dst_t), grp in edge_type_groups:
    total_before += len(grp)
    if len(grp) > MAX_EDGES_PER_RELATION:
        grp = grp.sample(n=MAX_EDGES_PER_RELATION, random_state=SEED)
    total_after += len(grp)
    src_local = grp['x_index'].map(lambda g: local_idx[g][1]).values
    dst_local = grp['y_index'].map(lambda g: local_idx[g][1]).values
    ei = torch.tensor(np.stack([src_local, dst_local]), dtype=torch.long)
    edge_index_by_type[(src_t, rel, dst_t)] = ei

edge_types = list(edge_index_by_type.keys())
edge_index_dict = {k: v.to(device) for k, v in edge_index_by_type.items()}
metadata = (node_types, edge_types)
assert type_counts['drug'] == n_drug and type_counts['disease'] == n_disease, 'Graph rebuild mismatch vs Phase 2 — investigate before trusting results.'
print(f'Edges before capping: {total_before} | after capping: {total_after}')
print('Rebuilt graph matches Phase 2 node counts. Node types:', type_counts)

Edges before capping: 8100498 | after capping: 1292656
Rebuilt graph matches Phase 2 node counts. Node types: {'anatomy': 14035, 'biological_process': 28642, 'cellular_component': 4176, 'disease': 17080, 'drug': 7957, 'effect/phenotype': 15311, 'exposure': 818, 'gene/protein': 27671, 'molecular_function': 11169, 'pathway': 2516}


In [22]:
# baselines/hgt_baseline.py equivalent
from torch_geometric.nn import HGTConv

class HGTBaseline(nn.Module):
    """2-layer HGT encoder + dot-product decoder, single-stage supervised training."""
    def __init__(self, node_types, metadata, type_counts, hidden=HIDDEN, num_heads=2, num_layers=2):
        super().__init__()
        self.embed = nn.ModuleDict({t: nn.Embedding(type_counts[t], hidden) for t in node_types})
        self.convs = nn.ModuleList([
            HGTConv(hidden, hidden, metadata, num_heads) for _ in range(num_layers)
        ])

    def forward(self, edge_index_dict):
        x_dict = {t: emb.weight for t, emb in self.embed.items()}
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {k: v.relu() for k, v in x_dict.items()}
        return x_dict

In [23]:
# baselines/han_baseline.py equivalent
from torch_geometric.nn import HANConv

class HANBaseline(nn.Module):
    """HAN encoder (attention over metapaths) + dot-product decoder. Same hidden dim as HGT."""
    def __init__(self, node_types, metadata, type_counts, hidden=HIDDEN, num_heads=2, num_layers=2):
        super().__init__()
        self.embed = nn.ModuleDict({t: nn.Embedding(type_counts[t], hidden) for t in node_types})
        self.convs = nn.ModuleList([
            HANConv(hidden, hidden, metadata, heads=num_heads) for _ in range(num_layers)
        ])

    def forward(self, edge_index_dict):
        x_dict = {t: emb.weight for t, emb in self.embed.items()}
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {k: v.relu() for k, v in x_dict.items()}
        return x_dict

In [24]:
# baselines/train_baseline.py equivalent — shared training/eval loop
import gc

def decode(emb_drug, emb_disease, drug_idx, disease_idx):
    return (emb_drug[drug_idx] * emb_disease[disease_idx]).sum(dim=-1)

def sample_negatives(pos_pairs, n_drug, n_disease, k=1):
    pos_set = set(map(tuple, pos_pairs))
    neg = []
    while len(neg) < len(pos_pairs) * k:
        d, s = rng.randint(0, n_drug), rng.randint(0, n_disease)
        if (d, s) not in pos_set:
            neg.append((d, s))
    return np.array(neg)

def bce_link_loss(z_dict, pos_pairs):
    neg_pairs = sample_negatives(pos_pairs, n_drug, n_disease)
    pos_idx = torch.tensor(pos_pairs, dtype=torch.long, device=device)
    neg_idx = torch.tensor(neg_pairs, dtype=torch.long, device=device)
    pos_score = decode(z_dict['drug'], z_dict['disease'], pos_idx[:, 0], pos_idx[:, 1])
    neg_score = decode(z_dict['drug'], z_dict['disease'], neg_idx[:, 0], neg_idx[:, 1])
    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)])
    return F.binary_cross_entropy_with_logits(scores, labels)

def evaluate(z_dict, pos_pairs):
    neg_pairs = sample_negatives(pos_pairs, n_drug, n_disease)
    pos_idx = torch.tensor(pos_pairs, dtype=torch.long, device=device)
    neg_idx = torch.tensor(neg_pairs, dtype=torch.long, device=device)
    with torch.no_grad():
        pos_score = torch.sigmoid(decode(z_dict['drug'], z_dict['disease'], pos_idx[:, 0], pos_idx[:, 1])).cpu().numpy()
        neg_score = torch.sigmoid(decode(z_dict['drug'], z_dict['disease'], neg_idx[:, 0], neg_idx[:, 1])).cpu().numpy()
    scores = np.concatenate([pos_score, neg_score])
    labels = np.concatenate([np.ones(len(pos_score)), np.zeros(len(neg_score))])
    return {'AUPRC': float(average_precision_score(labels, scores)), 'AUROC': float(roc_auc_score(labels, scores))}

def train_and_eval(model_class, model_name, split_name, ind_train, ind_valid, ind_test, contra_train, contra_valid, contra_test, n_epoch=100, lr=5e-4):
    # Free any leftover memory from a previous model/split before allocating a new one.
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = model_class(node_types, metadata, type_counts).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    param_count = sum(p.numel() for p in model.parameters())
    log = []
    use_amp = device.type == 'cuda'
    t0 = time.time()
    for epoch in range(n_epoch):
        model.train()
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            z_dict = model(edge_index_dict)
            loss_ind = bce_link_loss(z_dict, ind_train)
            loss_contra = bce_link_loss(z_dict, contra_train)
            loss = loss_ind + loss_contra
        opt.zero_grad(); loss.backward(); opt.step()
        log.append({'epoch': epoch, 'loss_indication': float(loss_ind.item()), 'loss_contraindication': float(loss_contra.item())})
        if epoch % 25 == 0:
            model.eval()
            with torch.no_grad():
                z_eval = model(edge_index_dict)
            valid_metrics = evaluate(z_eval, ind_valid)
            print(f'[{model_name}/{split_name}] epoch {epoch} loss={loss.item():.4f} valid_indication_AUPRC={valid_metrics["AUPRC"]:.4f}')
    train_time_min = (time.time() - t0) / 60
    gpu_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if device.type == 'cuda' else None

    model.eval()
    with torch.no_grad():
        z_final = model(edge_index_dict)
    ind_metrics = evaluate(z_final, ind_test)
    contra_metrics = evaluate(z_final, contra_test)

    pd.DataFrame(log).to_csv(f'{PROJECT_DIR}/results/{model_name}_{split_name}_training_log.csv', index=False)

    result = {
        'model': model_name.upper(),
        'type': 'Heterogeneous Graph Transformer (attention-based)' if model_name == 'hgt' else 'Heterogeneous Attention Network',
        'pretrained': False,
        'split': split_name,
        'seed': SEED,
        'indication_AUPRC': ind_metrics['AUPRC'],
        'indication_AUROC': ind_metrics['AUROC'],
        'contraindication_AUPRC': contra_metrics['AUPRC'],
        'contraindication_AUROC': contra_metrics['AUROC'],
        'training_time_min': round(train_time_min, 2),
        'param_count': param_count,
        'gpu_mem_peak_mb': round(gpu_mem_mb, 1) if gpu_mem_mb else None,
        'notes': f'Single-stage supervised training (no pretrain phase), same splits as Phase 2 (loaded from splits.npz), seed 42. Reduced scale to fit a single T4: hidden={HIDDEN}, edges capped at {MAX_EDGES_PER_RELATION}/relation, {n_epoch} epochs, mixed precision.'
    }
    with open(f'{PROJECT_DIR}/results/{model_name}_{split_name}.json', 'w') as f:
        json.dump(result, f, indent=2)
    print(json.dumps(result, indent=2))

    del model, opt
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return result

In [25]:
hgt_random = train_and_eval(HGTBaseline, 'hgt', 'random_split', ind_train_r, ind_valid_r, ind_test_r, contra_train_r, contra_valid_r, contra_test_r)

[hgt/random_split] epoch 0 loss=1.3853 valid_indication_AUPRC=0.5383
[hgt/random_split] epoch 25 loss=1.3617 valid_indication_AUPRC=0.9168
[hgt/random_split] epoch 50 loss=1.2700 valid_indication_AUPRC=0.9549
[hgt/random_split] epoch 75 loss=0.8908 valid_indication_AUPRC=0.9724
{
  "model": "HGT",
  "type": "Heterogeneous Graph Transformer (attention-based)",
  "pretrained": false,
  "split": "random_split",
  "seed": 42,
  "indication_AUPRC": 0.9833719351637628,
  "indication_AUROC": 0.9859309588919655,
  "contraindication_AUPRC": 0.9944982619187841,
  "contraindication_AUROC": 0.9957607523210908,
  "training_time_min": 1.54,
  "param_count": 4327100,
  "gpu_mem_peak_mb": 14644.0,
  "notes": "Single-stage supervised training (no pretrain phase), same splits as Phase 2 (loaded from splits.npz), seed 42. Reduced scale to fit a single T4: hidden=32, edges capped at 50000/relation, 100 epochs, mixed precision."
}


In [26]:
hgt_zeroshot = train_and_eval(HGTBaseline, 'hgt', 'zeroshot_split', ind_train_z, ind_valid_z, ind_test_z, contra_train_z, contra_valid_z, contra_test_z)

[hgt/zeroshot_split] epoch 0 loss=1.3855 valid_indication_AUPRC=0.5459
[hgt/zeroshot_split] epoch 25 loss=1.3598 valid_indication_AUPRC=0.9230
[hgt/zeroshot_split] epoch 50 loss=1.2524 valid_indication_AUPRC=0.9657
[hgt/zeroshot_split] epoch 75 loss=0.8678 valid_indication_AUPRC=0.9772
{
  "model": "HGT",
  "type": "Heterogeneous Graph Transformer (attention-based)",
  "pretrained": false,
  "split": "zeroshot_split",
  "seed": 42,
  "indication_AUPRC": 0.9768662655766696,
  "indication_AUROC": 0.9829931501982594,
  "contraindication_AUPRC": 0.9935181390268848,
  "contraindication_AUROC": 0.9947407862116066,
  "training_time_min": 1.49,
  "param_count": 4327100,
  "gpu_mem_peak_mb": 14643.3,
  "notes": "Single-stage supervised training (no pretrain phase), same splits as Phase 2 (loaded from splits.npz), seed 42. Reduced scale to fit a single T4: hidden=32, edges capped at 50000/relation, 100 epochs, mixed precision."
}


In [27]:
han_random = train_and_eval(HANBaseline, 'han', 'random_split', ind_train_r, ind_valid_r, ind_test_r, contra_train_r, contra_valid_r, contra_test_r)

[han/random_split] epoch 0 loss=1.3701 valid_indication_AUPRC=0.9690
[han/random_split] epoch 25 loss=1.3397 valid_indication_AUPRC=0.9708
[han/random_split] epoch 50 loss=1.2170 valid_indication_AUPRC=0.9736
[han/random_split] epoch 75 loss=0.9806 valid_indication_AUPRC=0.9729
{
  "model": "HAN",
  "type": "Heterogeneous Attention Network",
  "pretrained": false,
  "split": "random_split",
  "seed": 42,
  "indication_AUPRC": 0.9818071556711314,
  "indication_AUROC": 0.9860871080718548,
  "contraindication_AUPRC": 0.9812342987199284,
  "contraindication_AUROC": 0.9866273209255995,
  "training_time_min": 0.65,
  "param_count": 4169696,
  "gpu_mem_peak_mb": 14540.1,
  "notes": "Single-stage supervised training (no pretrain phase), same splits as Phase 2 (loaded from splits.npz), seed 42. Reduced scale to fit a single T4: hidden=32, edges capped at 50000/relation, 100 epochs, mixed precision."
}


In [28]:
han_zeroshot = train_and_eval(HANBaseline, 'han', 'zeroshot_split', ind_train_z, ind_valid_z, ind_test_z, contra_train_z, contra_valid_z, contra_test_z)

[han/zeroshot_split] epoch 0 loss=1.3678 valid_indication_AUPRC=0.9391
[han/zeroshot_split] epoch 25 loss=1.3342 valid_indication_AUPRC=0.9583
[han/zeroshot_split] epoch 50 loss=1.2097 valid_indication_AUPRC=0.9527
[han/zeroshot_split] epoch 75 loss=0.9830 valid_indication_AUPRC=0.9792
{
  "model": "HAN",
  "type": "Heterogeneous Attention Network",
  "pretrained": false,
  "split": "zeroshot_split",
  "seed": 42,
  "indication_AUPRC": 0.9853306871105153,
  "indication_AUROC": 0.9882659195833461,
  "contraindication_AUPRC": 0.9855060222981411,
  "contraindication_AUROC": 0.9903145439124498,
  "training_time_min": 0.64,
  "param_count": 4169696,
  "gpu_mem_peak_mb": 14540.1,
  "notes": "Single-stage supervised training (no pretrain phase), same splits as Phase 2 (loaded from splits.npz), seed 42. Reduced scale to fit a single T4: hidden=32, edges capped at 50000/relation, 100 epochs, mixed precision."
}


**Acceptance gate:** `hgt_random_split.json`, `hgt_zeroshot_split.json`, `han_random_split.json`, `han_zeroshot_split.json` all exist under `results/` on Drive, AUPRC/AUROC finite in [0, 1], param counts logged, training logs non-empty. Paste the four result dicts back before Phase 4 (aggregation needs all of Phase 2 + Phase 3's real numbers, not placeholders).